# Notebook 05 — Finding a Needle in 500M Rows
## Search Optimization for Fraud Investigations & Point Lookups

**What you'll learn**: When you need to find ONE specific transaction or search for a merchant pattern across 500M rows, there's a feature that makes it instant — but it only helps for "needle in a haystack" queries.

| | With Search Optimization | Without Search Optimization |
|---|---|---|
| **Find 1 transaction by ID** | Sub-second (reads 1-3 data blocks) | Full scan (reads everything) |
| **Search merchant name LIKE '%CRYPTO%'** | Fast (targeted blocks) | Full scan |
| **Broad filter (amount > 100)** | No help (returns too many rows) | Same speed — SOS can't help here |

In [ ]:
USE ROLE SYSADMIN;
USE WAREHOUSE HOL_XS;
USE SCHEMA JPMC_PERF_HOL.CONSUMER_BANKING;

ALTER SESSION SET USE_CACHED_RESULT = FALSE;

---
## Step 1: The Problem — Fraud Investigation Requires Full Table Scan

**Scenario**: A fraud investigator asks you to look up a suspicious transaction by its ID. With 500M transactions, this should be instant — but without Search Optimization, Snowflake reads EVERY row to find it.

In [ ]:
-- Point lookup without SOS — full scan of 500M rows for 1 record
SELECT *
FROM TRANSACTIONS
WHERE transaction_id = 'TXN-000250000000';

**Query Profile**: Snowflake scanned ALL partitions just to find 1 row! That's like reading every page of a phone book to find one name.

---
## Step 2: The Fix — Enable Search Optimization (Platform Team Action)

Your platform team enables a search index on the columns you need for lookups. This builds a behind-the-scenes index that tells Snowflake exactly where each value lives.

In [ ]:
-- Enable SOS on specific columns used for lookups
ALTER TABLE TRANSACTIONS ADD SEARCH OPTIMIZATION ON EQUALITY(transaction_id);
ALTER TABLE TRANSACTIONS ADD SEARCH OPTIMIZATION ON SUBSTRING(merchant_name);

-- Check SOS build progress
SHOW TABLES LIKE 'TRANSACTIONS';
-- Look at search_optimization_progress column

**Note**: SOS builds in the background. For a 500M-row table this may take several minutes.  
Wait for `search_optimization_progress = 100` before testing.

---
## Step 3: Best Case — Point Lookup WITH SOS

In [ ]:
-- Same query, now with SOS active — sub-second
SELECT *
FROM TRANSACTIONS
WHERE transaction_id = 'TXN-000250000000';

**Query Profile**: Partitions scanned drops to 1-3 (from thousands). Execution time: sub-second.

---
## Step 4: Best Case — Substring Search (Fraud Pattern)

**Scenario**: "Find all transactions at crypto exchanges" — substring search across 500M merchant names.

In [ ]:
-- Substring search WITH SOS — accelerated
SELECT transaction_date, account_id, amount, merchant_name
FROM TRANSACTIONS
WHERE merchant_name LIKE '%CRYPTO%'
  AND amount > 1000
ORDER BY amount DESC
LIMIT 20;

---
## Step 5: Worst Case — SOS on a Non-Selective Filter

**Scenario**: Broad filter that returns millions of rows — SOS provides zero benefit.

In [ ]:
-- WORST CASE: Low selectivity — returns ~60% of rows
-- SOS cannot help here (not a needle-in-haystack)
SELECT channel, COUNT(*), SUM(amount)
FROM TRANSACTIONS
WHERE amount > 100
GROUP BY 1;

**Query Profile**: Full scan regardless of SOS — the filter isn't selective enough.

---
## Step 6: Side-by-Side Metrics

In [ ]:
SELECT
    SUBSTR(query_text, 1, 60) AS query_preview,
    total_elapsed_time / 1000 AS elapsed_sec,
    query_id,
    bytes_scanned / (1024*1024*1024) AS gb_scanned
FROM TABLE(INFORMATION_SCHEMA.QUERY_HISTORY(
    END_TIME_RANGE_START => DATEADD(minute, -15, CURRENT_TIMESTAMP()),
    RESULT_LIMIT => 20
))
WHERE start_time > DATEADD(minute, -15, CURRENT_TIMESTAMP())
WHERE query_text ILIKE '%TRANSACTIONS%WHERE%'
  AND query_type = 'SELECT'
ORDER BY start_time DESC
LIMIT 4;

---
## Step 7: Cost Awareness

In [ ]:
-- What does SOS cost for this table?
SELECT SYSTEM$ESTIMATE_SEARCH_OPTIMIZATION_COSTS('TRANSACTIONS', 'EQUALITY(transaction_id)');

---
## Key Takeaways

| Query Pattern | Does SOS Help? | Example |
|---|---|---|
| Find one specific record by ID | **Yes** — instant | `WHERE transaction_id = 'TXN-000250000000'` |
| Search for a text pattern | **Yes** — fast | `WHERE merchant_name LIKE '%CRYPTO%'` |
| Broad filter returning many rows | **No** — same speed | `WHERE amount > 100` (returns millions) |

**Banking use cases for Search Optimization**:
- Fraud investigation: find a specific transaction instantly
- Compliance: locate customer by SSN hash
- Customer service: search merchant names for disputes
- AML: find transactions matching suspicious patterns

**💡 What to tell your platform team**:  
*"My fraud investigation queries search by `transaction_id` and `merchant_name LIKE` — these take 30+ seconds scanning the full table. Can we enable Search Optimization on these columns?"*

**How to know if you need it**:  
If your query returns a handful of rows but the Query Profile shows a full table scan → SOS is the answer.

**Next** → [Notebook 06 — Writing Faster Queries](./Notebook_06_Query_Tuning.ipynb)